In [1]:
%load_ext cudf.pandas

In [2]:
import sys, os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [3]:
%%RecordEvent
# %load_ext cudf.pandas
import numpy as np # linear algebra
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd

# Visualisation
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time

In [4]:
%%RecordEvent
%%time
### cell 0 ###

# load & cleanup
file = '/home/dias-benchmarks/notebooks/kkhandekar/environmental-vs-ai-startups-india-eda/input/indian-startup-recognized-by-dpiit/Startup_Counts_Across_India.csv'
df = pd.read_csv(file)

CPU times: user 11.4 ms, sys: 64 ms, total: 75.4 ms
Wall time: 70.9 ms


# -- STEFANOS -- Replicate Data

In [5]:
%%RecordEvent
%%time
### cell 1 ###

factor = 3000
df = pd.concat([df]*factor)
df.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 17673000 entries, 0 to 5890
Data columns (total 5 columns):
 #   Column    Dtype
---  ------    -----
 0   S No.     int64
 1   Year      int64
 2   State     object
 3   Industry  object
 4   Count     int64
dtypes: int64(3), object(2)
memory usage: 1.1+ GB
CPU times: user 1.84 s, sys: 350 ms, total: 2.19 s
Wall time: 2.11 s


In [6]:
%%RecordEvent
%%time
### cell 2 ###

df.drop('S No.',axis=1,inplace=True)
df.dropna(inplace=True)
df.reset_index(inplace=True,drop=True)

#view
df.head()

CPU times: user 25.7 ms, sys: 21.8 ms, total: 47.5 ms
Wall time: 41.5 ms


,Year,State,Industry,Count
0,2022,Andaman and Nicobar Islands,Agriculture,1
1,2022,Andaman and Nicobar Islands,AR VR (Augmented + Virtual Reality),1
2,2022,Andaman and Nicobar Islands,Construction,1
3,2022,Andaman and Nicobar Islands,Internet of Things,1
4,2022,Andaman and Nicobar Islands,Marketing,1


In [7]:
%%RecordEvent
%%time
### cell 3 ###

# Industry sub-categories for environmental & AI startups
env = ['Agriculture','Green Technology','Renewable Energy','Waste Management']
ai = ['AI','Robotics','Computer Vision']

# combined df - environmental & AI startups only
df_ea = df.loc[(df['Industry'].isin(env)) | (df['Industry'].isin(ai))].reset_index(drop=True,inplace=False)

# custom function to set Main Industry
def set_MainIndustry(ind):
    if ind in env:
        return 'ENV'
    else:
        return 'AI'

# adding a new column
df_ea['MainIndustry'] = df_ea.Industry.apply(lambda x: set_MainIndustry(x))

# basic stats
print(f"A total of {df_ea.shape[0]} startups were started in India between 2016 & 2022, out of which {df_ea.groupby('MainIndustry').size()['ENV']} are environmental related & {df_ea.groupby('MainIndustry').size()['AI']} are AI startups.")

A total of 2361000 startups were started in India between 2016 & 2022, out of which 1473000 are environmental related & 888000 are AI startups.
CPU times: user 1.54 s, sys: 52.7 ms, total: 1.59 s
Wall time: 1.58 s


In [8]:
%PrintCellInfo

======= Cell 0 =======
Input variables []
Active variables ['df']
Intermediate variables ['file']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables ['df']
Active variables ['df']
Intermediate variables ['factor']
Future variables []
Modified dataframes
======= Cell 2 =======
Input variables ['df']
Active variables ['df']
Intermediate variables []
Future variables []
Modified dataframes
  df
    Input columns: set()
    Changed columns: {'Year', 'Count', 'State', 'Industry'}
    Created columns: set()
    Deleted columns: {'S No.'}
======= Cell 3 =======
Input variables ['df']
Active variables []
Intermediate variables ['set_MainIndustry', 'ai', 'env', 'df_ea']
Future variables []
Modified dataframes
